# VerifAI — Treino (pares REAL × IA)

| Item | Valor |
|------|--------|
| REAL | `base/landscape_real/` (5k) |
| FAKE | `base/pares_ia/` (5k, mesma cena/pose) |
| Split | **4000** treino / **1000** teste (por par) |
| Entrada | Imagem inteira — letterbox 224 (HOG 128) |
| App | Veredito da **Fusão** |
| Saída | `cod/models/svm_v3_*.pkl` + scalers |


*Igor de Matos da Rosa · Nathália Geraldino Ribas*


## 1. Setup e caminhos


In [ ]:
# Setup
import random
import sys
import time
from collections import Counter
from pathlib import Path

import atexit
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

HERE = Path.cwd()
try:
    HERE_NB = Path(__vsc_ipynb_file__).resolve().parent
except NameError:
    HERE_NB = HERE

if (HERE / "main.ipynb").exists() or (HERE / "data.py").exists():
    COD_DIR = HERE
elif (HERE / "cod" / "main.ipynb").exists():
    COD_DIR = HERE / "cod"
elif (HERE_NB / "main.ipynb").exists():
    COD_DIR = HERE_NB
else:
    COD_DIR = HERE

ROOT_DIR = COD_DIR.parent if COD_DIR.name == "cod" else COD_DIR
if str(COD_DIR) not in sys.path:
    sys.path.insert(0, str(COD_DIR))

from data import (
    FEATURE_SIZE,
    HOG_SIZE,
    avisar_qualidade_bases,
    carregar_datasets_pares,
    carregar_imagem,
    resumo_rotulos,
)
from features import (
    TRANSFORMACOES,
    VARIANTES_TREINO,
    extrair_teste,
    extrair_teste_transformado,
    extrair_treino_aumentado,
)
from svm import treinar_svm_seguro

BASE_DIR = ROOT_DIR / "base"
MODELS_DIR = COD_DIR / "models"
RESULTS_DIR = COD_DIR / "results"
LOCAL_DIR = COD_DIR / "local"
for pasta in (MODELS_DIR, RESULTS_DIR, LOCAL_DIR):
    pasta.mkdir(parents=True, exist_ok=True)

atexit.register(lambda: plt.close("all"))
print("COD_DIR:", COD_DIR)
print("BASE_DIR:", BASE_DIR)
print("FEATURE_SIZE:", FEATURE_SIZE, "| HOG_SIZE:", HOG_SIZE)
print("landscape_real:", (BASE_DIR / "landscape_real").exists())
print("pares_ia:", (BASE_DIR / "pares_ia").exists())


## 2. Split de treino

| Pasta | Rótulo |
|-------|--------|
| `base/landscape_real` | REAL |
| `base/pares_ia` | FAKE |

| Treino | Teste |
|--------|-------|
| 4000 pares | 1000 pares |


In [ ]:
# Split fixo: 4000 treino / 1000 teste
CACHE_TAG = "v3pares"
N_TREINO, N_TESTE = 4000, 1000

train_c, train_y, test_c, test_y, meta = carregar_datasets_pares(
    BASE_DIR, seed=SEED, n_treino=N_TREINO, n_teste=N_TESTE
)
print("Meta:", meta)
for aviso in avisar_qualidade_bases(meta):
    print("⚠", aviso)
print(f"CACHE_TAG={CACHE_TAG}")
print("Treino:", len(train_c), resumo_rotulos(train_y))
print("Teste:", len(test_c), resumo_rotulos(test_y))
print("Amostras após aumento:", len(train_c) * len(VARIANTES_TREINO))
print("Variantes:", VARIANTES_TREINO)


## 3. Features aumentadas (FFT / HOG / LBP + fusão)


In [ ]:
# Extrai features com cache
tipos_features = ["fft", "hog", "lbp"]
features, scalers = {}, {}

for tipo in tipos_features:
    print(f"\n=== {tipo.upper()} ===")
    X_train, y_train = extrair_treino_aumentado(
        train_c, train_y, tipo, LOCAL_DIR, tag=CACHE_TAG
    )
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test, y_test = extrair_teste(test_c, test_y, tipo, LOCAL_DIR, tag=CACHE_TAG)
    X_test_s = scaler.transform(X_test)
    features[tipo] = {
        "X_train": X_train_s,
        "y_train": y_train,
        "X_test": X_test_s,
        "y_test": y_test,
    }
    scalers[tipo] = scaler
    joblib.dump(scaler, MODELS_DIR / f"scaler_v3_{tipo}.pkl")
    print("Treino:", features[tipo]["X_train"].shape, "| Teste:", features[tipo]["X_test"].shape)

y_ref = features["fft"]["y_train"]
y_test_ref = features["fft"]["y_test"]
features["fusion"] = {
    "X_train": np.hstack([features[t]["X_train"] for t in tipos_features]),
    "y_train": y_ref,
    "X_test": np.hstack([features[t]["X_test"] for t in tipos_features]),
    "y_test": y_test_ref,
}
print("FUSION:", features["fusion"]["X_train"].shape)
print("Scalers salvos em", MODELS_DIR)


## 4. Treinar SVM (RBF + peso em REAL)

| Item | Detalhe |
|------|--------|
| Backend | ThunderSVM → sklearn RBF → RBF aproximado |
| `PESO_REAL` | 2.0 = menos falso positivo em foto real |
| Saída | `svm_v3_*.pkl` — o app usa a **Fusão** |


In [ ]:
# Treino SVM — peso_real alto = menos falso positivo no app
PREFER_GPU = True
PESO_REAL = 2.0
baseline_results, tabela = {}, []

def treinar_familia(nome, X_train, y_train, X_test, y_test):
    print(f"\n=== SVM [{nome}] ===")
    t0 = time.time()
    modelo, backend = treinar_svm_seguro(
        X_train,
        y_train,
        prefer_gpu=PREFER_GPU,
        C=10.0,
        seed=SEED,
        peso_real=PESO_REAL,
    )
    caminho = MODELS_DIR / f"svm_v3_{nome.lower()}.pkl"
    joblib.dump(modelo, caminho)
    dt = time.time() - t0
    y_pred = modelo.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, labels=[0, 1], zero_division=0
    )
    print(f"  backend={backend} acc={acc:.4f} tempo={dt:.1f}s → {caminho.name}")
    print(f"  FAKE  prec={prec[0]:.3f} rec={rec[0]:.3f} | REAL prec={prec[1]:.3f} rec={rec[1]:.3f}")
    print(classification_report(y_test, y_pred, target_names=["FAKE", "REAL"], digits=3))
    return {
        "modelo": modelo,
        "acuracia": acc,
        "backend": backend,
        "y_pred": y_pred,
        "tempo": dt,
        "prec_fake": float(prec[0]),
        "rec_real": float(rec[1]),
    }

for tipo in tipos_features:
    baseline_results[tipo] = treinar_familia(
        tipo.upper(),
        features[tipo]["X_train"],
        features[tipo]["y_train"],
        features[tipo]["X_test"],
        features[tipo]["y_test"],
    )
    r = baseline_results[tipo]
    tabela.append({
        "feature": tipo.upper(),
        "backend": r["backend"],
        "n_treino": len(features[tipo]["y_train"]),
        "acc": f"{r['acuracia']:.4f}",
        "prec_fake": f"{r['prec_fake']:.3f}",
        "rec_real": f"{r['rec_real']:.3f}",
        "tempo_s": f"{r['tempo']:.1f}",
    })

baseline_results["fusion"] = treinar_familia(
    "FUSION",
    features["fusion"]["X_train"],
    features["fusion"]["y_train"],
    features["fusion"]["X_test"],
    features["fusion"]["y_test"],
)
r = baseline_results["fusion"]
tabela.append({
    "feature": "FUSION",
    "backend": r["backend"],
    "n_treino": len(features["fusion"]["y_train"]),
    "acc": f"{r['acuracia']:.4f}",
    "prec_fake": f"{r['prec_fake']:.3f}",
    "rec_real": f"{r['rec_real']:.3f}",
    "tempo_s": f"{r['tempo']:.1f}",
})
df_baseline = pd.DataFrame(tabela)
print("\n" + df_baseline.to_string(index=False))
print("\nFoque em FUSION (modelo do app).")
df_baseline.to_csv(RESULTS_DIR / "metrics_v3_baseline.csv", index=False)


## 5. Robustez às transformações


In [ ]:
# Avalia cada transformação no teste
resultados = []
for nome_transf in TRANSFORMACOES:
    print(f"\n--- {nome_transf} ---")
    X_por_tipo = {}
    for tipo in tipos_features:
        X_raw, y_raw = extrair_teste_transformado(
            test_c, test_y, tipo, nome_transf, LOCAL_DIR, tag=CACHE_TAG
        )
        X_por_tipo[tipo] = scalers[tipo].transform(X_raw)
        y_transf = y_raw
    X_por_tipo["fusion"] = np.hstack([X_por_tipo[t] for t in tipos_features])
    for tipo in list(tipos_features) + ["fusion"]:
        y_pred = baseline_results[tipo]["modelo"].predict(X_por_tipo[tipo])
        acc = accuracy_score(y_transf, y_pred)
        base = baseline_results[tipo]["acuracia"]
        resultados.append({
            "transformacao": nome_transf,
            "feature": tipo,
            "acc": acc,
            "acc_orig": base,
            "queda": base - acc,
        })
        print(f"  {tipo:7s} acc={acc:.4f} queda={base - acc:+.4f}")

df_rob = pd.DataFrame(resultados)
df_rob.to_csv(RESULTS_DIR / "metrics_v3_robustez.csv", index=False)
print("\nSalvo metrics_v3_robustez.csv")
print(df_rob.groupby("feature")["queda"].mean().sort_values())


## 6. Gráficos


In [ ]:
# Barras e heatmap
ordem = list(TRANSFORMACOES.keys())
fig, ax = plt.subplots(figsize=(12, 5))
for tipo in list(tipos_features) + ["fusion"]:
    sub = df_rob[df_rob["feature"] == tipo].set_index("transformacao").loc[ordem]
    ax.plot(ordem, sub["acc"], marker="o", label=tipo)
    ax.axhline(baseline_results[tipo]["acuracia"], linestyle="--", alpha=0.4)
ax.set_title("Acurácia por transformação")
ax.set_ylabel("Acurácia")
ax.tick_params(axis="x", rotation=45)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "v3_acc_por_transf.png", dpi=150)
plt.show()

pivot = df_rob.pivot(index="transformacao", columns="feature", values="queda").loc[ordem]
plt.figure(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="Reds")
plt.title("Queda de acurácia")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "v3_heatmap_queda.png", dpi=150)
plt.show()
print("FUSION acc original:", f"{baseline_results['fusion']['acuracia']:.4f}")
print("Backend:", baseline_results["fusion"]["backend"])
